In [1]:
!pip install underthesea tokenizers -q 
!pip install pandarallel -q 
!pip install pyarrow -q 
!pip install evaluate rouge_score -q
!pip install torchinfo -q 
!pip install pyngrok -q
!pip install bert_score -q
!pip install ninja packaging --quiet
!pip install causal-conv1d --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 40.4 MB/s eta 0:00:0

In [2]:
from underthesea import word_tokenize
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
import pandas as pd
import os

# Standard library
import os
import random
import math
import copy
import multiprocessing
import time
import subprocess

# Data & utils
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

# Underthesea & tokenizers
from underthesea import word_tokenize
from pandarallel import pandarallel
from pandarallel.core import WorkerStatus
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder

# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

# PyTorch AMP
# from torch.cuda.amp import autocast, GradScaler

# Scheduler
from torch.optim.lr_scheduler import LambdaLR

# TensorBoard
from torch.utils.tensorboard import SummaryWriter

# torchinfo
from torchinfo import summary

# Ngrok & Kaggle
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient


def segment_text(text):
  try:
    return word_tokenize(text, format='text')
  except:
    return ""

if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_data_compounded_with_ner.parquet'):
  train_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_data_compounded_with_ner.parquet')
else:
  train_df = df.dropna()
  train_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train-00000-of-00001.parquet')
  train_df = train_df.dropna()
  train_df['article'] = train_df['article'].parallel_apply(segment_text)
  train_df['summary'] = train_df['summary'].parallel_apply(segment_text)
  train_df.to_parquet('train_data_compounded.parquet')
  print('Done')

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


2026-06-10 08:58:12.487593: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781081892.877984      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781081892.984568      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781081893.951441      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781081893.951484      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781081893.951487      23 computation_placer.cc:177] computation placer alr

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder

if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_summarization_tokenizer.json'):
  tokenizer = Tokenizer.from_file('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/train_summarization_tokenizer.json')
else:
    tokenizer = Tokenizer(BPE(unk_token='<UNK>'))
    
    tokenizer.pre_tokenizer = Metaspace(replacement=" ", prepend_scheme="always")
    
    tokenizer.decoder = MetaspaceDecoder(replacement=" ", prepend_scheme="always")
    
    trainer = BpeTrainer(
        vocab_size=36000,
        min_frequency=2,
        special_tokens=[
            '<PAD>',
            '<UNK>',
            '<BOS>',
            '<EOS>'
        ]
    )
    
    text_list = train_df['article'].to_list() + train_df['summary'].to_list()
    tokenizer.train_from_iterator(text_list, trainer)
    tokenizer.save('train_summarization_tokenizer.json')
    print('Done')

In [4]:
cau_test = "Tôi đang chạy thử thuật toán BPE với cụm từ Xyz_Abc_9999 và VinFast_VF8_Pro."

encoded_test = tokenizer.encode(cau_test)
print(encoded_test.tokens)

[' Tôi', ' đang', ' chạy', ' thử', ' thuật', ' toán', ' B', 'P', 'E', ' với', ' cụm', ' từ', ' X', 'y', 'z', '_Ab', 'c_', '99', '99', ' và', ' VinFas', 't_V', 'F', '8', '_Pro', '.']


# Phải có dấu câu vì trong thực tế việc đặt dấu câu nó cũng sẽ ảnh hưởng đến ý nghĩa của câu đó.

In [5]:
import pandas as pd
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

# validate
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/val_data_compounded.parquet'):
  val_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/val_data_compounded.parquet')
else:
  val_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-dataset/valid-00000-of-00001.parquet')
  val_df = val_df.dropna()
  val_df['article'] = val_df['article'].parallel_apply(segment_text)
  val_df['summary'] = val_df['summary'].parallel_apply(segment_text)
  val_df.to_parquet('val_data_compounded.parquet')
  print('Done')

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [6]:
# validate
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-test-set/test_data_compounded.parquet'):
  test_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-test-set/test_data_compounded.parquet')
else:
  test_df = pd.read_parquet('/kaggle/input/datasets/longnguyen2k5/nlp-summarization-test-set/test-00000-of-00001 (1).parquet')
  test_df = val_df.dropna()
  test_df['article'] = test_df['article'].parallel_apply(segment_text)
  test_df['summary'] = test_df['summary'].parallel_apply(segment_text)
  test_df.to_parquet('test_data_compounded.parquet')
  print('Done')

In [7]:
import torch
from torch.nn.utils.rnn import pad_sequence

val_ids = [encoding.ids for encoding in tokenizer.encode_batch(val_df['article'].to_list() + val_df['summary'].to_list())]
val_ids = pad_sequence([torch.tensor(ids) for ids in val_ids], batch_first=True, padding_value=tokenizer.token_to_id('<PAD>'))
count_unk = val_ids == tokenizer.token_to_id('<UNK>')
padding_mask = val_ids == tokenizer.token_to_id('<PAD>')
print(count_unk.sum().item())
print((count_unk.sum() / (count_unk.numel() - padding_mask.sum())).item())

5
6.806221335864393e-06


In [8]:

if train_df is not None and val_df is not None:
  train_df['article_ids'] = train_df['article'].apply(lambda x: tokenizer.encode(x).ids)
  train_df['summary_ids'] = train_df['summary'].apply(lambda x: tokenizer.encode(x).ids)
  val_df['article_ids'] = val_df['article'].apply(lambda x: tokenizer.encode(x).ids)
  val_df['summary_ids'] = val_df['summary'].apply(lambda x: tokenizer.encode(x).ids)

if test_df is not None: 
  test_df['article_ids'] = test_df['article'].apply(lambda x: tokenizer.encode(x).ids)
  test_df['summary_ids'] = test_df['summary'].apply(lambda x: tokenizer.encode(x).ids)

Vaidate thì chỉ xuất hiện 5 UNK token có nghĩa là tokenizer hoạt động tốt trên tập validate

In [9]:
# Reprocedure
import random
import numpy as np
import os

def seed(seed_value=42):
  os.environ['PYTHONHASHSEED'] = str(seed_value)
  torch.manual_seed(seed_value)
  random.seed(seed_value)
  np.random.seed(seed_value)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
  print('Done')

def seed_worker(worker_id):
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

In [10]:
from pandarallel.core import WorkerStatus
## START PROGRAM

seed(42)

from torch.utils.data import Dataset, DataLoader
import os 
import multiprocessing

num_gpus = torch.cuda.device_count()
num_cores = multiprocessing.cpu_count()

active_devices = max(1, num_gpus)

BOS = tokenizer.token_to_id('<BOS>')
EOS = tokenizer.token_to_id('<EOS>')
BATCH_SIZE = 8 * active_devices
OPTIMAL_WORKERS = num_cores
MAX_SEQ_LEN=1024
MAX_SUM_LEN=360


class SummarizationDataset(Dataset):
  def __init__(self, df):
    self.df = df
    self.has_ner = 'src_ner_mask' in df.columns and 'tgt_ner_mask' in df.columns

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    article_ids = self.df.iloc[idx]['article_ids']
    summary_ids = self.df.iloc[idx]['summary_ids']
      
    row = self.df.iloc[idx]
    
    src_ner_mask = row['src_ner_mask'] if self.has_ner else []
    tgt_ner_mask = row['tgt_ner_mask'] if self.has_ner else []
    
    return article_ids, summary_ids, src_ner_mask, tgt_ner_mask

def collate_fn(batch):
  article_tensors = []
  summary_tensors = []
  src_ner_tensors = []
  tgt_ner_tensors = []
  
  PAD_ID = tokenizer.token_to_id('<PAD>')

  for article_ids, summary_ids, src_ner, tgt_ner in batch:
    article_sliced = article_ids[:MAX_SEQ_LEN]
    article_tensors.append(torch.tensor(article_sliced))

    summary_processed = [BOS] + summary_ids[:MAX_SUM_LEN-2] + [EOS]
    summary_tensors.append(torch.tensor(summary_processed))

    if len(src_ner) > 0:
      src_ner_tensors.append(torch.tensor(src_ner[:MAX_SEQ_LEN], dtype=torch.float32))
    else:
      src_ner_tensors.append(torch.zeros(len(article_sliced), dtype=torch.float32))

    if len(tgt_ner) > 0:
      tgt_ner_processed = [0.0] + tgt_ner[:MAX_SUM_LEN-2] + [0.0]
      tgt_ner_tensors.append(torch.tensor(tgt_ner_processed, dtype=torch.float32))
    else:
      tgt_ner_tensors.append(torch.zeros(len(summary_processed), dtype=torch.float32))

  article_tensors = pad_sequence(article_tensors, batch_first=True, padding_value=PAD_ID)
  summary_tensors = pad_sequence(summary_tensors, batch_first=True, padding_value=PAD_ID)
  
  src_ner_tensors = pad_sequence(src_ner_tensors, batch_first=True, padding_value=0.0)
  tgt_ner_tensors = pad_sequence(tgt_ner_tensors, batch_first=True, padding_value=0.0)

  return article_tensors, summary_tensors, src_ner_tensors, tgt_ner_tensors


g = torch.Generator()
g.manual_seed(42)
train_dataset = SummarizationDataset(train_df)
train_dataloader = DataLoader(train_dataset, 
                              batch_size=BATCH_SIZE, 
                              shuffle=True, 
                              collate_fn=collate_fn,
                              worker_init_fn=seed_worker, 
                              generator=g, 
                              pin_memory=True, 
                              persistent_workers=(OPTIMAL_WORKERS > 0),
                              num_workers=OPTIMAL_WORKERS)

validate_dataset = SummarizationDataset(val_df)
validate_dataloader = DataLoader(validate_dataset, 
                                 batch_size=BATCH_SIZE, 
                                 shuffle=False, 
                                 collate_fn=collate_fn, 
                                 pin_memory=True, 
                                 persistent_workers=(OPTIMAL_WORKERS > 0),
                                 num_workers=OPTIMAL_WORKERS)

if test_df is not None: 
    test_dataset = SummarizationDataset(test_df)

Done


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from torch.optim import Adam, AdamW
from torch.utils.tensorboard import SummaryWriter
from torch.amp import GradScaler, autocast 
import math
import copy
from tqdm import tqdm
import evaluate
import pandas as pd
from IPython.display import display # Dùng để in bảng đẹp trên Jupyter/Kaggle

import warnings
import transformers

# Tắt cảnh báo màu đỏ của Python
warnings.filterwarnings("ignore")

# Ép HuggingFace chỉ in ra lỗi nghiêm trọng (Tắt cái bảng LOAD REPORT đi)
transformers.logging.set_verbosity_error()

class Embeddings(nn.Module):
  def __init__(self, d_model, vocab):
    super().__init__()
    self.lut = nn.Embedding(vocab, d_model)
    self.d_model = d_model

  def forward(self, x):
    return self.lut(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, dropout, max_len=2000):
    super().__init__()
    position = torch.arange(0, max_len).unsqueeze(-1) # max_len, 1
    pe = torch.zeros(max_len, d_model) # max_len, d_model
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.)) / d_model) # d_model,
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe.unsqueeze_(0) # 1, max_len, d_model
    self.register_buffer('pe', pe)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, step=0):
    x = x + self.pe[:, step: step + x.size(1)].requires_grad_(False)
    return self.dropout(x)


def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
  d_k = query.size(-1)
  scores = torch.matmul(query, key.transpose(-2,-1)) / math.sqrt(d_k)
  if mask is not None:
    min_value = torch.finfo(scores.dtype).min
    scores = scores.masked_fill(mask == 0, min_value)
  p_attn = F.softmax(scores.float(), dim=-1).to(scores.dtype)
  if dropout is not None:
    p_attn = dropout(p_attn)
  return torch.matmul(p_attn, value), p_attn

class MultiHeadAttention(nn.Module):
  def __init__(self, h, d_model, dropout=0.1):
    super().__init__()
    assert d_model % h == 0
    self.d_k = d_model // h
    self.h = h
    self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
    self.dropout = nn.Dropout(dropout)

  def forward(self, query, key, value, mask=None, past_key_value=None, use_cache=False, is_cross_attention=False):
    batch_size = query.size(0)
    # batch_size, h, seq_len, d_k
    if past_key_value is not None:
        if is_cross_attention: 
            query = self.linears[0](query).view(batch_size, -1, self.h, self.d_k).transpose(1,2)
            key, value = past_key_value
        else: 
            query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (query, key, value))]
            K, V = past_key_value
            key = torch.cat([K, key], dim=-2)
            value = torch.cat([V, value], dim=-2)
    else: 
        query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (query, key, value))]
        
    present_key_value = (key, value) if use_cache else None

    if mask is not None:
        mask = mask.unsqueeze(1)
    x, attn = scaled_dot_product_attention(query, key, value, mask=mask, dropout=self.dropout)
    x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)

    return self.linears[-1](x), present_key_value

class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()
    self.linear_1 = nn.Linear(d_model, d_ff)
    self.dropout = nn.Dropout(dropout)
    self.linear_2 = nn.Linear(d_ff, d_model)
    self.activation = nn.ReLU()

  def forward(self, x):
    return self.linear_2(self.dropout(self.activation(self.linear_1(x))))

class LayerNorm(nn.Module):
  def __init__(self, d_model, eps=1e-6):
    super().__init__()
    self.a_2 = nn.Parameter(torch.ones(d_model))
    self.b_2 = nn.Parameter(torch.zeros(d_model))
    self.eps = eps

  def forward(self, x):
    x_f32 = x.float()
    mean = torch.mean(x_f32, dim=-1, keepdim=True)
    var = torch.var(x_f32, dim=-1, keepdim=True, unbiased=False)
      
    out = (x_f32 - mean) / torch.sqrt(var + self.eps)
    return (out.to(x.dtype)) * self.a_2 + self.b_2


class ResidualConnection(nn.Module):
  def __init__(self, d_model, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.layer_norm = LayerNorm(d_model)

  def forward(self, x, sublayer):
    return x + self.dropout(sublayer(self.layer_norm(x)))

def clones(module, N):
  return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class EncoderLayer(nn.Module):
  def __init__(self, d_model, d_ff, h, dropout=0.1):
    super().__init__()
    self.self_attn = MultiHeadAttention(h, d_model, dropout=dropout)
    self.feed_forward = FeedForward(d_model, d_ff, dropout=dropout)
    self.sublayer = clones(ResidualConnection(d_model, dropout=dropout), 2)
    self.d_model = d_model

  def forward(self, x, mask):
    x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask=mask)[0])
    x = self.sublayer[1](x, self.feed_forward)
    return x

class Encoder(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.layers = clones(EncoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.embedding = Embeddings(d_model, tokenizer.get_vocab_size())
    self.pe = PositionalEncoding(d_model, dropout=dropout)

  def forward(self, x, mask=None):
    x = self.embedding(x)
    x = self.pe(x)
    for layer in self.layers:
      x = layer(x, mask)
    return x

class DecoderLayer(nn.Module):
  def __init__(self, d_model, d_ff, h, dropout=0.1):
    super().__init__()
    self.masked_attn = MultiHeadAttention(h, d_model, dropout=dropout)
    self.attn = MultiHeadAttention(h, d_model, dropout=dropout)
    self.feed_forward = FeedForward(d_model, d_ff, dropout=dropout)
    self.sublayer = clones(ResidualConnection(d_model, dropout=dropout), 3)
    self.d_model = d_model

  def forward(self, x, memory, src_mask, tgt_mask, past_kv=None, use_cache=False):
    past_self_kv = past_kv[0] if past_kv is not None else None
    past_cross_kv = past_kv[1] if past_kv is not None else None        
    present_self_kv = None
    def self_attn_wrapper(q): 
        nonlocal present_self_kv 
        out, present_self_kv = self.masked_attn(q, q, q, mask=tgt_mask, past_key_value=past_self_kv, use_cache=use_cache, is_cross_attention=False)
        return out
    x = self.sublayer[0](x, self_attn_wrapper)

    present_cross_kv = None
    def cross_attn_wrapper(q): 
        nonlocal present_cross_kv
        out, present_cross_kv = self.attn(q, memory, memory, mask=src_mask, past_key_value=past_cross_kv, use_cache=use_cache, is_cross_attention=True)
        return out
    x = self.sublayer[1](x, cross_attn_wrapper)
    x = self.sublayer[2](x, self.feed_forward)
    return x, (present_self_kv, present_cross_kv)

class Decoder(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.layers = clones(DecoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.embedding = Embeddings(d_model, tokenizer.get_vocab_size())
    self.pe = PositionalEncoding(d_model, dropout=dropout)
    self.linear = nn.Linear(d_model, tokenizer.get_vocab_size())

  def forward(self, x, memory, src_mask, tgt_mask, past_kvs=None, use_cache=False, step=0):
    x = self.embedding(x)
    x = self.pe(x, step=step)
    present_kvs = [] 
    for i, layer in enumerate(self.layers):
      past_kv = past_kvs[i] if past_kvs is not None else None
      x, present_kv = layer(x, memory, src_mask, tgt_mask, past_kv=past_kv, use_cache=use_cache)
      present_kvs.append(present_kv)
    return self.linear(x), present_kvs

class BaselineTransformer(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.encoder = Encoder(d_model, d_ff, h, N, dropout=dropout)
    self.decoder = Decoder(d_model, d_ff, h, N, dropout=dropout)
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True, step=step)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens
              
              

In [12]:
# Count for create histogram
from tqdm import tqdm
def generate_penalty(alpha=1.0):
    _dataloader = DataLoader(train_dataset, 
                              batch_size=BATCH_SIZE, 
                              shuffle=False, 
                              collate_fn=collate_fn,
                              pin_memory=False, 
                              num_workers=0)
    VOCAB_SIZE = tokenizer.get_vocab_size()
    global_counts = torch.zeros(VOCAB_SIZE, dtype=torch.long, device='cpu')
    
    for article_ids, summary_ids, _, _ in tqdm(_dataloader): 
        text_ids = torch.cat([article_ids, summary_ids], dim=-1)
        tokens = text_ids.view(-1).cpu()
        batch_counts = torch.bincount(tokens, minlength=VOCAB_SIZE)
        global_counts += batch_counts
    
    global_counts = torch.clamp(global_counts, min=1)
    max_count = global_counts.max().float()
    
    penalty_tensor = alpha*(1.0 - (torch.log(global_counts.float()) / torch.log(max_count)))
    return penalty_tensor.unsqueeze(0) # 1, vocab_size

In [13]:
class RoPECache(nn.Module): 
    def __init__(self, d_k, dropout=0.1, max_len=2048): 
        super().__init__()
        position = torch.arange(0, max_len) # max_len,
        inv_freq = torch.exp(torch.arange(0, d_k, 2) * (-math.log(10000.)) / d_k) # d_k/2 ,
        theta = torch.einsum('i,j->ij', position, inv_freq) # max_len, d_k/2
        emb = torch.cat([theta, theta],dim=-1) # max_len, d_k
        # persistent for not save in .pt
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    def forward(self, seq_len, step=0): 
        return self.cos_cached[step:step+seq_len].unsqueeze(0).unsqueeze(0), self.sin_cached[step:step+seq_len].unsqueeze(0).unsqueeze(0) # 1, 1, seq_len, d_model

def rotate_half(x): 
    x1 = x[..., :x.shape[-1]//2]
    x2 = x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin): 
    q_embed = q * cos + rotate_half(q) * sin
    k_embed = k * cos + rotate_half(k) * sin

    return q_embed, k_embed

class SwiGLU(nn.Module):
    def __init__(self, d_ff, d_model, dropout=0.1): 
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x): 
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        return self.w_down(gate * up)

class RMSNorm(nn.Module): 
    def __init__(self, d_model, eps=1e-6): 
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x): 
        x_f32 = x.float()
        rms = torch.sqrt(x_f32.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms.to(x.dtype)) * self.g

class ResidualConnectionWithRMS(nn.Module): 
    def __init__(self, d_model, dropout=0.1): 
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer): 
        return x + self.dropout(sublayer(self.norm(x)))
        

class MultiHeadAttentionWithRoPE(nn.Module): 
    def __init__(self, h, d_model, dropout=0.1): 
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
        self.dropout = nn.Dropout(dropout)
        self.rope_cache = RoPECache(self.d_k)

    def forward(self, q, k, v, mask=None, past_kv=None, use_cache=None, is_cross_attention=False): 
        batch_size = q.size(0)

        if past_kv is not None: 
            if (is_cross_attention): 
                query = self.linears[0](q).view(batch_size, -1, self.h, self.d_k).transpose(1,2)
                key, value = past_kv
            else: 
                query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (q, k, v))]
                seq_len = query.shape[-2]
                K, V = past_kv 
                step = K.shape[-2]
                cos, sin = self.rope_cache(seq_len, step=step)
                query, key = apply_rotary_pos_emb(query, key, cos, sin)
                key = torch.cat([K, key], dim=-2)
                value = torch.cat([V, value], dim=-2)
        else:
            # batch_size, h, seq_len, d_k
            query, key, value = [linear(x).view(batch_size, -1, self.h, self.d_k).transpose(1,2) for linear, x in zip(self.linears[:3], (q, k, v))]
            if is_cross_attention == False: 
                seq_len = query.shape[-2]
                cos, sin = self.rope_cache(seq_len)
                query, key = apply_rotary_pos_emb(query, key, cos, sin)

        persent_kv = (key, value) if use_cache == True else None
        
        if mask is not None:
            mask = mask.unsqueeze(1)
        x, attn = scaled_dot_product_attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)
    
        return self.linears[-1](x), persent_kv

class ImprovedEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.swiglu = SwiGLU(d_ff,d_model)
        self.residual = clones(ResidualConnectionWithRMS(d_model),2)

    def forward(self, x, mask): 
        x = self.residual[0](x, lambda x: self.attn(x,x,x, mask=mask)[0])
        x = self.residual[1](x, self.swiglu)
        return x

class ImprovedEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.encoder_layers = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        
    def forward(self, src_ids, mask=None): 
        x = self.embedding(src_ids) 
        for layer in self.encoder_layers:
            x = layer(x,mask)
        return x


class ImprovedDecoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.masked_attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.cross_attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.sublayer = clones(ResidualConnectionWithRMS(d_model), 3)
        self.swiglu = SwiGLU(d_ff, d_model)

    def forward(self, x, memory, src_mask, tgt_mask, past_kv=None, use_cache=False):
        past_self_kv = past_kv[0] if past_kv is not None else None
        past_cross_kv = past_kv[1] if past_kv is not None else None        
        present_self_kv = None
        def self_attn_wrapper(q): 
            nonlocal present_self_kv 
            out, present_self_kv = self.masked_attn(q, q, q, mask=tgt_mask, past_kv=past_self_kv, use_cache=use_cache, is_cross_attention=False)            
            return out
        x = self.sublayer[0](x, self_attn_wrapper)
    
        present_cross_kv = None
        def cross_attn_wrapper(q): 
            nonlocal present_cross_kv
            out, present_cross_kv = self.cross_attn(q, memory, memory, mask=src_mask, past_kv=past_cross_kv, use_cache=use_cache, is_cross_attention=True)            
            return out
        x = self.sublayer[1](x, cross_attn_wrapper)
        x = self.sublayer[2](x, self.swiglu)
        return x, (present_self_kv, present_cross_kv)

class ImprovedDecoder(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
    self.layers = clones(ImprovedDecoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.linear = nn.Linear(d_model, tokenizer.get_vocab_size())
      
  def forward(self, x, memory, src_mask, tgt_mask, past_kvs=None, use_cache=False):
    x = self.embedding(x)
    present_kvs = [] 
    for i, layer in enumerate(self.layers):
      past_kv = past_kvs[i] if past_kvs is not None else None
      x, present_kv = layer(x, memory, src_mask, tgt_mask, past_kv=past_kv, use_cache=use_cache)
      present_kvs.append(present_kv)
    return self.linear(x), present_kvs

class ImprovedBaselineTransformer(nn.Module):
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.encoder = ImprovedEncoder(d_model, d_ff, h, N, dropout=dropout)
    self.decoder = ImprovedDecoder(d_model, d_ff, h, N, dropout=dropout)
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens

In [14]:
from mamba_ssm import Mamba
class BiMambaBlock(nn.Module): 
    def __init__(self, d_model, d_state=16, d_conv=4, expand=1): 
        super().__init__()
        self.mamba_forward = Mamba(
            d_model=d_model,
            d_state=d_state, # Mặc định tốt nhất
            d_conv=d_conv,   # Mặc định tốt nhất
            expand=expand    # Ép về 1 để công bằng tham số với Transformer Baseline
        )
        self.mamba_backward = Mamba(
            d_model=d_model,
            d_state=d_state, # Mặc định tốt nhất
            d_conv=d_conv,   # Mặc định tốt nhất
            expand=expand    # Ép về 1 để công bằng tham số với Transformer Baseline
        )

    def forward(self, x): 
        # batch_size, seq_len, d_model
        x_forward = self.mamba_forward(x)
        x_flipped = torch.flip(x, dims=[-2]).contiguous()
        x_backward = self.mamba_backward(x_flipped)
        x_backward = torch.flip(x_backward, dims=[-2]).contiguous()
        return x_forward, x_backward

class BiMambaEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff_new, dropout=0.1): 
        super().__init__()
        
        self.bi_mamba = BiMambaBlock(d_model=d_model, expand=1) 
        
        self.swiglu = SwiGLU(d_ff_new, d_model)
        
        self.pipe_gate = nn.Linear(d_model, 1)
        self.pipe_semantic = nn.Linear(d_model, d_model)

        self.gate_fusion = nn.Linear(2,1)  
        
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        self.eps = 1e-6
        

    def forward(self, x, mask, min_p=0.1): 
        # mask: batch_size, 1, seq_len
        mask = mask.to(x.dtype)
        mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
        
        x_normed = self.norm(x)
        x_masked = x_normed * mask # batch_size, seq_len, d_model
        x_masked = x_masked + (1.0 - mask) * self.eps # Tránh Nan 
        
        x_forward, x_backward = self.bi_mamba(x_masked)

        x_forward = x_forward * mask
        x_backward = x_backward * mask

        x_forward = self.swiglu(x_forward)
        x_backward = self.swiglu(x_backward)
        
        gate_fwd_logits = self.pipe_gate(x_forward) # batch_size, seq_len, 1
        gate_bwd_logits = self.pipe_gate(x_backward) # batch_size, seq_len, 1

        combined_logits = torch.cat([gate_fwd_logits, gate_bwd_logits], dim=-1) # batch_size, seq_len, 2
        combined_mask = torch.sigmoid(self.gate_fusion(combined_logits)) * mask # batch_size, seq_len, 1  
        
        sem_fwd = self.pipe_semantic(x_forward) * mask 
        sem_bwd = self.pipe_semantic(x_backward) * mask
        
        combined_sem = sem_fwd + sem_bwd

        gated_sem = combined_sem * combined_mask

        final_sem = combined_sem + (gated_sem - gated_sem.detach())

        x = x + self.dropout(final_sem)

        mask_loss = (combined_mask * mask).sum() / (mask.sum() + self.eps)
        
        mask_bool = (combined_mask.squeeze(-1) > min_p) # batch_size, seq_len
        max_k = mask_bool.sum(-1).max().item() 
        max_k = max(1, max_k) 

        topk_scores, topk_indices = torch.topk(combined_mask.squeeze(-1), max_k, dim=1)
        gather_idx = topk_indices.unsqueeze(-1).expand(-1, -1, x.size(-1))
        compressed_x = torch.gather(x, dim=1, index=gather_idx) # batch_size, seq_len, d_model
        
        decoder_padding_mask = torch.gather(mask.squeeze(-1), dim=1, index=topk_indices).bool()
        decoder_padding_mask[:, 0] = True
        return compressed_x, decoder_padding_mask , mask_loss

class BiMambaEncoder(nn.Module): 
    def __init__(self, d_model, d_ff,d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.bi_mamba_block = BiMambaEncoderLayer(d_model, d_ff_new, dropout=dropout)

    def forward(self, x, mask=None, min_p=0.1): 
        x = self.embedding(x) # batch_size, seq_len, d_model
        # mask: batch_size, 1, seq_len
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x,mask)
        output, new_mask, mask_loss = self.bi_mamba_block(x,mask, min_p = min_p)
                                                          
        new_mask = new_mask.unsqueeze(-2)  # (batch, seq_len, 1) -> (batch, 1, seq_len)
        return output, new_mask, mask_loss

class HyperSphereLinear(nn.Module): 
    def __init__(self, d_model, vocab_size, initial_gamma=10.0, tied_weight=None): 
        super().__init__()
        if tied_weight is not None:
            self.weight = tied_weight
        else:
            self.weight = nn.Parameter(torch.empty(vocab_size, d_model))
            nn.init.xavier_uniform_(self.weight)
        self.gamma = nn.Parameter(torch.tensor(initial_gamma))
        self.eps = 1e-8
        self.register_buffer('output_scale', torch.tensor(1.0))

    def forward(self, x): 
        x_norm = F.normalize(x, p=2, dim=-1, eps=self.eps)
        w_norm = F.normalize(self.weight, p=2, dim=-1, eps=self.eps)
        gamma_clamped = torch.clamp(self.gamma, min=0.1, max=50.0)
        cosine_sim = F.linear(x_norm, w_norm)  
        logits = cosine_sim * gamma_clamped
        logits = torch.clamp(logits, min=-11.0, max=11.0)
        return logits



class HyperSphereTransformerDecoder(nn.Module): 
  def __init__(self, d_model, d_ff, h, N, dropout=0.1, tied_weight=None):
    super().__init__()
    self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
    self.layers = clones(ImprovedDecoderLayer(d_model, d_ff, h, dropout=dropout), N)
    self.linear = HyperSphereLinear(d_model, tokenizer.get_vocab_size(), 
                                     tied_weight=tied_weight)     
      
  def forward(self, x, memory, src_mask, tgt_mask, past_kvs=None, use_cache=False):
    x = self.embedding(x)
    present_kvs = [] 
    for i, layer in enumerate(self.layers):
      past_kv = past_kvs[i] if past_kvs is not None else None
      x, present_kv = layer(x, memory, src_mask, tgt_mask, past_kv=past_kv, use_cache=use_cache)
      present_kvs.append(present_kv)
    return self.linear(x), present_kvs


class TransformerWithSoftPromptMamba(nn.Module):
  def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
    super().__init__()
    self.encoder = BiMambaEncoder(d_model, d_ff,d_ff_new, h, N-1, dropout=dropout)
    self.decoder = HyperSphereTransformerDecoder(d_model, d_ff, h, N, dropout=dropout)
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory, src_mask, mask_loss = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output, mask_loss

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory, src_mask, _ = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens

In [15]:
# 1. LAYER MỚI: Kế thừa lại toàn bộ các tầng tuyến tính của Layer cũ
class EntityGatedBiMambaEncoderLayer(BiMambaEncoderLayer): 
    def forward(self, x, mask, src_ner_mask=None, min_p=0.1): 
        # mask: batch_size, 1, seq_len
        mask = mask.to(x.dtype)
        mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
        
        x_normed = self.norm(x)
        x_masked = x_normed * mask 
        x_masked = x_masked + (1.0 - mask) * self.eps 
        
        x_forward, x_backward = self.bi_mamba(x_masked)

        x_forward = x_forward * mask
        x_backward = x_backward * mask

        x_forward = self.swiglu(x_forward)
        x_backward = self.swiglu(x_backward)
        
        gate_fwd_logits = self.pipe_gate(x_forward) 
        gate_bwd_logits = self.pipe_gate(x_backward) 

        combined_logits = torch.cat([gate_fwd_logits, gate_bwd_logits], dim=-1) 
        combined_mask = torch.sigmoid(self.gate_fusion(combined_logits)) * mask   
        
        sem_fwd = self.pipe_semantic(x_forward) * mask 
        sem_bwd = self.pipe_semantic(x_backward) * mask
        
        combined_sem = sem_fwd + sem_bwd
        gated_sem = combined_sem * combined_mask
        final_sem = combined_sem + (gated_sem - gated_sem.detach())

        x = x + self.dropout(final_sem)

        base_mask_loss = (combined_mask * mask).sum() / (mask.sum() + self.eps)
        
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            ner_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='none')
            actual_ner_loss = (ner_loss * target_ner).sum() / (target_ner.sum() + self.eps)
            
            total_mask_loss = base_mask_loss + actual_ner_loss
        else:
            total_mask_loss = base_mask_loss

        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        mask_bool = (combined_mask.squeeze(-1) > min_p) 
        max_k = mask_bool.sum(-1).max().item() 
        max_k = max(1, max_k) 

        topk_scores, topk_indices = torch.topk(combined_mask.squeeze(-1), max_k, dim=1)
        gather_idx = topk_indices.unsqueeze(-1).expand(-1, -1, x.size(-1))
        compressed_x = torch.gather(x, dim=1, index=gather_idx) 
        
        decoder_padding_mask = torch.gather(mask.squeeze(-1), dim=1, index=topk_indices).bool()
        decoder_padding_mask[:, 0] = True
        
        return compressed_x, decoder_padding_mask, total_mask_loss

class EntityGatedBiMambaEncoder(BiMambaEncoder):
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__(d_model, d_ff, d_ff_new, h, N, dropout=dropout)
        self.bi_mamba_block = EntityGatedBiMambaEncoderLayer(d_model, d_ff_new, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None, min_p=0.1): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, new_mask, mask_loss = self.bi_mamba_block(x, mask, src_ner_mask=src_ner_mask, min_p=min_p)
                                                                    
        new_mask = new_mask.unsqueeze(-2)  
        return output, new_mask, mask_loss

class EntityGatedMambaSeq2Seq(TransformerWithSoftPromptMamba):
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__(d_model, d_ff, d_ff_new, h, N, dropout=dropout)
        self.encoder = EntityGatedBiMambaEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, src_mask, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, src_mask, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens
    

In [16]:
class EntityGuidedTransformerEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.swiglu = SwiGLU(d_ff, d_model)
        self.residual = clones(ResidualConnectionWithRMS(d_model), 2)
        
        self.entity_predictor = nn.Linear(d_model, 1)
        self.prob_to_prompt = nn.Linear(1, d_model, bias=False)
        
    def forward(self, x, mask, src_ner_mask=None):
        combined_logits = self.entity_predictor(x)
        combined_mask = torch.sigmoid(combined_logits) * mask.to(x.dtype).squeeze(-2).unsqueeze(-1)
        
        mask_loss = torch.tensor(0.0, device=x.device)
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            mask_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='mean')
            
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        prompt_vector = self.prob_to_prompt(combined_mask)
        x_injected = x + prompt_vector
        
        x = self.residual[0](x_injected, lambda x: self.attn(x, x, x, mask=mask)[0])
        x = self.residual[1](x, self.swiglu)
        
        return x, mask_loss



class EntityGuidedPureTransformerEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.entity_guided_block = EntityGuidedTransformerEncoderLayer(d_model, d_ff_new,h, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, mask_loss = self.entity_guided_block(x, mask, src_ner_mask=src_ner_mask)
                                          
        return output, mask_loss
        
class EntityGuidedPureTransformer(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__()
        self.encoder = EntityGuidedPureTransformerEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens

In [17]:
class TransformerWithWeightTyingAndHyperSphereNorm(nn.Module): 
  def __init__(self, d_model, d_ff, h, N, dropout=0.1):
    super().__init__()
    self.encoder = ImprovedEncoder(d_model, d_ff, h, N, dropout=dropout)
    self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
    self.decoder.embedding.weight = self.encoder.embedding.weight
    self.d_model = d_model

  def forward(self, src, tgt, src_mask=None, tgt_mask=None):
    memory = self.encoder(src, src_mask)
    output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
    return output

  @torch.inference_mode()
  def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
    device = next(self.parameters()).device
    device_type=device.type
    src_token = src_token.to(device)
    src_mask = src_mask.to(device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
      memory = self.encoder(src_token, src_mask)

    if strategy == 'greedy': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
    elif strategy == 'greedy_with_penalty': 
        return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

  def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
      device = next(self.parameters()).device
      device_type=device.type
      batch_size = src_token.size(0)
      
      tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
      unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)

      past_kvs = None

      if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
      
      for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
      return tgt_tokens

In [18]:
class EntityGuidedBiMambaEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff_new, dropout=0.1): 
        super().__init__()
        
        self.bi_mamba = BiMambaBlock(d_model=d_model, expand=1) 
        self.swiglu = SwiGLU(d_ff_new, d_model)
        
        self.entity_predictor = nn.Linear(d_model, 1)
        self.pipe_semantic = nn.Linear(d_model, d_model)
        
        self.prob_to_prompt = nn.Linear(1, d_model, bias=False)
        
        self.norm = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        self.eps = 1e-6
        
    def forward(self, x, mask, src_ner_mask=None, min_p=0.3): 
        # mask: batch_size, 1, seq_len
        mask = mask.to(x.dtype)
        mask = mask.squeeze(-2).unsqueeze(-1) # batch_size, seq_len, 1
        
        combined_logits = self.entity_predictor(x)
        combined_mask = torch.sigmoid(combined_logits) * mask
        
        mask_loss = torch.tensor(0.0, device=x.device)
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            mask_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='mean')
            
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        prompt_vector = self.prob_to_prompt(combined_mask)
        x_injected = x + prompt_vector
        
        x_normed = self.norm(x_injected)
        x_masked = x_normed * mask 
        x_masked = x_masked + (1.0 - mask) * self.eps 
        
        x_forward, x_backward = self.bi_mamba(x_masked)

        x_forward = x_forward * mask
        x_backward = x_backward * mask

        x_forward = self.swiglu(x_forward)
        x_backward = self.swiglu(x_backward)
        
        sem_fwd = self.pipe_semantic(x_forward) * mask 
        sem_bwd = self.pipe_semantic(x_backward) * mask
        combined_sem = sem_fwd + sem_bwd

        output = x_injected + self.dropout(combined_sem)
        
        return output, mask_loss

class EntityGuidedBiMambaEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.bi_mamba_block = EntityGuidedBiMambaEncoderLayer(d_model, d_ff_new, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None, min_p=0.3): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, mask_loss = self.bi_mamba_block(x, mask, src_ner_mask=src_ner_mask, min_p=min_p)
                                          
        return output, mask_loss
        
class EntityGuidedHybridMamba(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__()
        self.encoder = EntityGuidedBiMambaEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens

In [19]:
class EntityGuidedTransformerEncoderLayer(nn.Module): 
    def __init__(self, d_model, d_ff, h, dropout=0.1): 
        super().__init__()
        self.attn = MultiHeadAttentionWithRoPE(h, d_model)
        self.swiglu = SwiGLU(d_ff, d_model)
        self.residual = clones(ResidualConnectionWithRMS(d_model), 2)
        
        self.entity_predictor = nn.Linear(d_model, 1)
        self.prob_to_prompt = nn.Linear(1, d_model, bias=False)
        
    def forward(self, x, mask, src_ner_mask=None):
        combined_logits = self.entity_predictor(x)
        combined_mask = torch.sigmoid(combined_logits) * mask.to(x.dtype).squeeze(-2).unsqueeze(-1)
        
        mask_loss = torch.tensor(0.0, device=x.device)
        if src_ner_mask is not None and src_ner_mask.sum() > 0:
            target_ner = src_ner_mask.unsqueeze(-1).to(x.dtype)
            combined_mask_clamped = combined_mask.clamp(1e-6, 1 - 1e-6)
            mask_loss = F.binary_cross_entropy(combined_mask_clamped, target_ner, reduction='mean')
            
            ner_bool = (src_ner_mask > 0).unsqueeze(-1).bool()
            combined_mask = torch.where(ner_bool, torch.ones_like(combined_mask), combined_mask)
            
        prompt_vector = self.prob_to_prompt(combined_mask)
        x_injected = x + prompt_vector
        
        x = self.residual[0](x_injected, lambda x: self.attn(x, x, x, mask=mask)[0])
        x = self.residual[1](x, self.swiglu)
        
        return x, mask_loss



class EntityGuidedPureTransformerEncoder(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1): 
        super().__init__()
        self.embedding = nn.Embedding(tokenizer.get_vocab_size(), d_model)
        self.attn_encode_layer = clones(ImprovedEncoderLayer(d_model,d_ff,h, dropout=dropout), N)
        self.entity_guided_block = EntityGuidedTransformerEncoderLayer(d_model, d_ff_new,h, dropout=dropout)

    def forward(self, x, mask=None, src_ner_mask=None): 
        x = self.embedding(x) 
        if mask is not None: 
            tmp_mask = mask.squeeze(-2).unsqueeze(-1) 
            x = x * tmp_mask
        for layer in self.attn_encode_layer: 
            x = layer(x, mask)
            
        output, mask_loss = self.entity_guided_block(x, mask, src_ner_mask=src_ner_mask)
                                          
        return output, mask_loss
        
class EntityGuidedPureTransformer(nn.Module): 
    def __init__(self, d_model, d_ff, d_ff_new, h, N, dropout=0.1):
        super().__init__()
        self.encoder = EntityGuidedPureTransformerEncoder(d_model, d_ff, d_ff_new, h, N-1, dropout=dropout)
        self.decoder = HyperSphereTransformerDecoder(
            d_model, d_ff, h, N, 
            dropout=dropout,
            tied_weight=self.encoder.embedding.weight)
        self.decoder.embedding.weight = self.encoder.embedding.weight
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_ner_mask=None):
        memory, mask_loss = self.encoder(src, src_mask, src_ner_mask=src_ner_mask)
        output = self.decoder(tgt, memory, src_mask, tgt_mask)[0]
        
        return output, mask_loss
        
    @torch.inference_mode()
    def generate_summary(self, src_token, src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0, **kwarg):
        device = next(self.parameters()).device
        device_type=device.type
        src_token = src_token.to(device)
        src_mask = src_mask.to(device)
        with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
          memory, _ = self.encoder(src_token, src_mask)
    
        if strategy == 'greedy': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len)
        elif strategy == 'greedy_with_penalty': 
            return self._greedy_search(src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor, base_penalty=base_penalty)

    def _greedy_search(self, src_token, src_mask, memory, bos_idx, eos_idx, pad_idx, max_len, penalty_tensor=None, base_penalty=0.0): 
        device = next(self.parameters()).device
        device_type=device.type
        batch_size = src_token.size(0)
        
        tgt_tokens = torch.full((batch_size,1), bos_idx, dtype=torch.long, device=device)
        unfinished = torch.ones((batch_size,1), dtype=torch.bool, device=device)
        
        past_kvs = None
        
        if penalty_tensor is not None:
          # penaldy_tensor shape: 1,vocab_size
          penalty_tensor = penalty_tensor.to(device)
          vocab_size = penalty_tensor.size(1)
          counts = torch.zeros((batch_size, vocab_size), dtype=torch.long, device=device)
        
        for step in range(max_len): 
          with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type=='cuda')): 
              input_token = tgt_tokens if step == 0 else tgt_tokens[:, -1:]
              output, present_kvs = self.decoder(input_token, memory, src_mask, None, past_kvs=past_kvs, use_cache=True)
          next_token_logits = output[:, -1, :] # batch_size, vocab_size
          if penalty_tensor is not None: 
              mask = counts > 0
              next_token_logits = next_token_logits - (penalty_tensor * counts + mask * base_penalty)
          next_token = next_token_logits.argmax(dim= -1).unsqueeze(-1) # batch_size, 1
          next_token = next_token * unfinished + (~unfinished) * pad_idx
          tgt_tokens = torch.cat([tgt_tokens, next_token], dim=-1)
          unfinished = unfinished & (eos_idx != next_token)
          past_kvs = present_kvs
          if penalty_tensor is not None: 
              batch_indices = torch.arange(batch_size, device=device)
              active_mask = unfinished.squeeze(-1)
              active_indices = batch_indices[active_mask]
              active_tokens = next_token.squeeze(-1)[active_mask]
              counts[active_indices, active_tokens] += 1
          if unfinished.max() == 0: 
              break
        return tgt_tokens

In [20]:
def subsequence_mask(size):
  attn_shape = (1, size, size)
  return torch.tril(torch.ones(attn_shape).type(torch.bool))

class Batch:
  def __init__(self, src, tgt=None, pad_idx=0, device='cpu'):
    self.src = src.to(device)
    tgt = tgt.to(device) if tgt is not None else tgt
    self.src_mask = (self.src != pad_idx).unsqueeze(-2) # batch_size, 1, seq_len
    if tgt is not None:
      self.tgt = tgt[:, :-1]
      self.tgt_y = tgt[:, 1:]
      self.tgt_mask = self.make_std_mask(self.tgt, pad_idx).to(device)
      self.ntokens = (self.tgt_y != pad_idx).data.sum()

  @staticmethod
  def make_std_mask(tgt, pad):
    tgt_mask = (tgt != pad).unsqueeze(-2) # batch_size, 1, seq_len
    tgt_submask = subsequence_mask(tgt.size(-1)).to(tgt_mask.device) # 1, seq_len, seq_len
    return tgt_mask & tgt_submask # batch_size, seq_len, seq_len

def train_step(model, optimizer, criterion, scheduler, train_dataloader,pad_idx, epoch_num, scaler, model_name='baseline_transformer'):
  model.train()
  device = next(model.parameters()).device
  device_type = device.type
  total_loss = 0
  train_bar = tqdm(train_dataloader, desc=f'Epoch {epoch_num} [TRAIN]')
    # for src_ids, tgt_ids, src_ner_tensors, tgt_ner_tensors in train_dataloader:
  for src_ids, tgt_ids, src_ner_tensors, tgt_ner_tensors  in train_bar:
    batch = Batch(src_ids, tgt_ids, pad_idx, device=device)
    optimizer.zero_grad()
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')):
        if 'entity' in model_name: 
            src_ner_gpu = src_ner_tensors.to(device)
            result = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask, src_ner_gpu)
        else: 
            result = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        if isinstance(result, tuple):
            output, mask_loss = result
            ce_loss = criterion(output.contiguous().view(-1, output.size(-1)), batch.tgt_y.contiguous().view(-1))
            LAMBDA_SPARSITY = 0.05
            if mask_loss.dim() > 0: 
                mask_loss = mask_loss.mean()
            loss = ce_loss + LAMBDA_SPARSITY * mask_loss
        else: 
            output = result # batch_size, seq_len, vocab_size
            loss = criterion(output.contiguous().view(-1, output.size(-1)), batch.tgt_y.contiguous().view(-1))
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
      
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
      
    scheduler.step()
    total_loss += loss.item()
  return total_loss / len(train_dataloader)

@torch.inference_mode()
def validate_step(model, criterion, validate_dataloader, pad_idx, epoch_num, model_name='baseline_transformer'):
  model.eval()
  device = next(model.parameters()).device
  device_type = device.type
  total_loss = 0
  validate_bar = tqdm(validate_dataloader, desc=f'Epoch {epoch_num} [VALIDATE]')
  for src_ids, tgt_ids, _ , _ in validate_bar:
    batch = Batch(src_ids, tgt_ids, pad_idx, device=device)
    with autocast(device_type=device_type, dtype=torch.float16, enabled=(device_type == 'cuda')): 
        result = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        if isinstance(result, tuple):
            output, _ = result
        else:
            output = result
        loss = criterion(output.contiguous().view(-1, output.size(-1)), batch.tgt_y.contiguous().view(-1))
    total_loss += loss.item()
  return total_loss / len(validate_dataloader)

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, min_lr_ratio=0.1):
    def lr_lambda(current_step):
        # 1. Giai đoạn Warmup (Tăng dần từ 0 lên 1)
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        
        # 2. Giai đoạn Cosine Decay (Giảm dần từ 1 về min_lr_ratio)
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        
        # Công thức hàm Cosine hạ cánh mềm
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        
        # Tính toán hệ số nhân cuối cùng (đảm bảo không rớt xuống mức 0 tuyệt đối)
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

    # LambdaLR sẽ lấy `lr` ban đầu của AdamW nhân với kết quả của hàm lr_lambda ở trên
    return LambdaLR(optimizer, lr_lambda) 


def get_lr_multiplier(step_num, d_model=512, warmup_steps=4000):
    # Tránh step_num = 0 gây lỗi chia cho 0
    step_num = max(step_num, 1)
    return (d_model ** -0.5) * min(step_num ** -0.5, step_num * (warmup_steps ** -1.5))


def train_loop(model, optimizer, criterion, scheduler, train_dataloader, val_dataloader, pad_idx, epoch, best_val_loss=None, start_epoch=0, model_name='baseline_transformer'):
  model.train()
  patience = 10
  non_improve_count = 0
  best_val_loss = float('inf') if best_val_loss is None else best_val_loss
  is_cuda = torch.cuda.is_available()
  scaler = GradScaler('cuda', enabled=is_cuda)
  writer = SummaryWriter(f'/kaggle/working/runs/{model_name}')
  for e in range(start_epoch, start_epoch + epoch):
    train_epoch_loss = train_step(model, optimizer, criterion, scheduler, train_dataloader, pad_idx, e, scaler, model_name=model_name)
    val_epoch_loss = validate_step(model, criterion, val_dataloader, pad_idx, e, model_name=model_name)
    writer.add_scalar('Loss/train', train_epoch_loss, e)
    writer.add_scalar('Loss/val', val_epoch_loss, e)

    if val_epoch_loss < best_val_loss:
      non_improve_count = 0
      best_val_loss = val_epoch_loss
      checkpoint = {
        'epoch': e,
        'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss
      }
      save_dir = f'/kaggle/working/{model_name}'
      if not os.path.exists(save_dir):
          os.makedirs(save_dir)
      torch.save(checkpoint, f'/kaggle/working/{model_name}/best_checkpoint.pt')  
    else:
      non_improve_count += 1

    if non_improve_count >= patience:
      break

  writer.close()

In [21]:
# import time
# import subprocess
# from pyngrok import ngrok
# from kaggle_secrets import UserSecretsClient

# # 1. Lấy token bảo mật
# user_secrets = UserSecretsClient()
# my_secret_token = user_secrets.get_secret("NGROK_TOKEN")
# ngrok.set_auth_token(my_secret_token)

# # Xóa các tunnel cũ nếu bạn lỡ chạy lại cell này nhiều lần (Tránh lỗi kẹt Port)
# ngrok.kill()

# # 2. Dùng subprocess để ép TensorBoard chạy ngầm (Bypass luật của Kaggle)
# print("Đang khởi động TensorBoard...")
# subprocess.Popen(['tensorboard', '--logdir', '/kaggle/working/runs', '--host', '0.0.0.0', '--port', '6006'])

# # Chờ 3 giây cho server load xong
# time.sleep(3)

# # 3. Tạo đường hầm Ngrok
# tunnel = ngrok.connect(6006)
# print("🚀 Đã xong! Bấm vào link này để xem TensorBoard:")
# print(tunnel.public_url)

    # checkpoint = {
    #     'epoch': e,
    #     'model_state_dict': model.state_dict(),
    #     'optimizer_state_dict': optimizer.state_dict(),
    #     'scheduler_state_dict': scheduler.state_dict(),
    #     'best_val_loss': best_val_loss
    #   }

In [22]:
def main_train(model, epochs, model_name='baseline_transformer'): 
    total_steps = len(train_dataloader) * epochs 

    warmup_steps = int(0.1 * total_steps) 
    
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=1.0,               
        betas=(0.9, 0.98),    
        eps=1e-5,             
        weight_decay=0.01     
    )
    # scheduler = get_cosine_schedule_with_warmup(
    #     optimizer, 
    #     num_warmup_steps=warmup_steps, 
    #     num_training_steps=total_steps,
    #     min_lr_ratio=0.1  
    # )
    scheduler = LambdaLR(optimizer, lr_lambda=lambda step: get_lr_multiplier(step, d_model=512, warmup_steps=4000))
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id('<PAD>'))
    if os.path.exists(f'/kaggle/working/{model_name}/best_checkpoint.pt'):
        checkpoint = torch.load(f'/kaggle/working/{model_name}/best_checkpoint.pt', map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        if torch.cuda.device_count() > 1:
            print(f"🔥 Kích hoạt chạy song song trên {torch.cuda.device_count()} GPUs!")
            model = nn.DataParallel(model)
        
        model = model.to(device)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
        remain_epoch = epochs - checkpoint['epoch'] -1 
        start_epoch=checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        
        train_loop(model, optimizer, criterion, scheduler, train_dataloader, validate_dataloader, tokenizer.token_to_id('<PAD>'), remain_epoch, 
                   best_val_loss=best_val_loss, 
                   start_epoch=start_epoch,
                  model_name=model_name)
    else: 
        train_loop(model, optimizer, criterion, scheduler, train_dataloader, validate_dataloader, tokenizer.token_to_id('<PAD>'), epochs, model_name=model_name)

In [23]:
def calculate_intra_distinct(predictions, n_gram=2): 
    if len(predictions) == 0: 
        return 0.0
    distinct_score = 0.
    for text in predictions: 
        words = text.split()
        tup_list = [tuple(words[i:i+n_gram]) for i in range(len(words) - n_gram + 1)]
        distinct = set(tup_list)
        if len(tup_list) > 0: 
            distinct_score += len(distinct) / len(tup_list)
    return distinct_score / len(predictions)
    
def calculate_inter_distinct(predictions, n_gram=2): 
    distinct = set()
    total_nums = 0
    for text in predictions: 
        words = text.split()
        tup_list = [tuple(words[i:i+n_gram]) for i in range(len(words) - n_gram + 1)]
        total_nums += len(tup_list)
        distinct.update(tup_list)
    return len(distinct) / total_nums

def calculate_length_ratio(predictions, references):
    total_ratio = 0.0
    for pred, ref in zip(predictions, references):
        len_pred = len(pred.split())
        len_ref = len(ref.split())
        # Tránh chia cho 0 nếu reference bị rỗng
        if len_ref > 0:
            total_ratio += len_pred / len_ref
    return total_ratio / len(predictions) if len(predictions) > 0 else 0.0


def calculate_novel_ngrams_score(predictions, sources, n_gram=1):
    if len(predictions) == 0:
        return 0.0
    
    total_novel_ratio = 0.0
    for pred, src in zip(predictions, sources):
        pred_words = pred.split()
        src_words = src.split()
        
        src_ngrams = set([
            tuple(src_words[i:i+n_gram]) 
            for i in range(len(src_words) - n_gram + 1)
        ])
        
        pred_ngrams = [
            tuple(pred_words[i:i+n_gram]) 
            for i in range(len(pred_words) - n_gram + 1)
        ]
        
        if len(pred_ngrams) == 0:
            continue
            
        novel_count = sum(1 for tg in pred_ngrams if tg not in src_ngrams)
        
        total_novel_ratio += novel_count / len(pred_ngrams)
        
    return (total_novel_ratio / len(predictions)) * 100

def calculate_evalute_metrics(predictions, references): 
  final_results = {}
  intra_distinct = calculate_intra_distinct(predictions)
  inter_distinct = calculate_inter_distinct(predictions)
  len_ratio = calculate_length_ratio(predictions, references)
    
  final_results['intra_distinct'] = intra_distinct
  final_results['inter_distinct'] = inter_distinct
  final_results['len_ratio'] = len_ratio
    
  for metric in metrics_list:
    print(f"{metric.name.upper()}")

    if metric.name in ["bertscore", "bert_score"]:
      raw_bert = metric.compute(predictions=predictions, references=references, lang="vi")
      final_results["bertscore"] = {
          "precision": sum(raw_bert["precision"]) / len(raw_bert["precision"]),
          "recall": sum(raw_bert["recall"]) / len(raw_bert["recall"]),
          "f1": sum(raw_bert["f1"]) / len(raw_bert["f1"])
      }
    else:
      final_results[metric.name] = metric.compute(predictions=predictions, references=references)

  return final_results
    
def evaluate_model(model, test_dataloader, bos_idx, eos_idx, pad_idx, tokenizer, strategy='greedy', penalty_tensor=None, base_penalty=0.0):
  actual_model = model.module if isinstance(model, nn.DataParallel) else model
  actual_model.eval()
  device = next(model.parameters()).device

  all_predictions = []
  all_references = []
  all_sources = []

  test_bar = tqdm(test_dataloader, desc='[GENERATING PREDICTIONS]')
  for src_ids, tgt_ids, _, _ in test_bar:
    batch = Batch(src_ids, tgt_ids, pad_idx, device=device)
    prediction = actual_model.generate_summary(batch.src, batch.src_mask, bos_idx, eos_idx, pad_idx, max_len=360, strategy=strategy, penalty_tensor=penalty_tensor, base_penalty=base_penalty)
    
    prediction = tokenizer.decode_batch(prediction.cpu().tolist(), skip_special_tokens=True)
    references = tokenizer.decode_batch(batch.tgt_y.cpu().tolist(), skip_special_tokens=True)
    sources = tokenizer.decode_batch(batch.src.cpu().tolist(), skip_special_tokens=True) # Giải mã nguồn gốc
    
    clean_prediction = [text.replace('_', ' ') for text in prediction]
    clean_references = [text.replace('_', ' ') for text in references]
    clean_sources = [text.replace('_', ' ') for text in sources]
    
    all_predictions.extend(clean_prediction)
    all_references.extend(clean_references)
    all_sources.extend(clean_sources)
      
  return all_sources, all_references, all_predictions
    
metrics_list = [
evaluate.load('rouge'),
evaluate.load('bleu'),
evaluate.load('bertscore')
]

# Test
# BOS = tokenizer.token_to_id('<BOS>')
# EOS = tokenizer.token_to_id('<EOS>')
# PAD = tokenizer.token_to_id('<PAD>')
# tmp_src_ids, tmp_tgt_ids, _ , _ = next(iter(validate_dataloader))

# device = 'cuda' if torch.cuda.is_available() else 'cpu' 
# D_MODEL = 512
# # D_FF = 2048
# D_FF = 1365 # 2048 * 2 / 3
# D_FF_NEW = 798
# H = 8
# N = 6
# EPOCHS= 100 if device == 'cuda' else 1

# actual_model = EntityGuidedPureTransformer(D_MODEL, D_FF, D_FF,H, N+5)
# if os.path.exists('/kaggle/input/datasets/longnguyen2k5/entity-guided-pure-transformer-results/entity_guided_pure_transformer/best_checkpoint.pt'): 
#   checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/entity-guided-pure-transformer-results/entity_guided_pure_transformer/best_checkpoint.pt', map_location=device)
#   actual_model.load_state_dict(checkpoint['model_state_dict'])
# actual_model = actual_model.to(device)
# actual_model.eval()
# batch = Batch(tmp_src_ids, tmp_tgt_ids, PAD, device=device)


# def eval_with_miniset(model, batch, bos_idx, eos_idx, pad_idx, tokenizer): 
#     model.eval()
#     results_log = [] 
    
#     no_penalty_predictions = model.generate_summary(batch.src, batch.src_mask, bos_idx, eos_idx, pad_idx,  max_len=360, strategy='greedy',penalty_tensor=None, base_penalty=0.0)
#     no_penalty_predictions = tokenizer.decode_batch(no_penalty_predictions.cpu().tolist(), skip_special_tokens=True)
#     no_penalty_predictions = [text.replace('_', ' ') for text in no_penalty_predictions]
#     references = tokenizer.decode_batch(batch.tgt_y.cpu().tolist(), skip_special_tokens=True)
#     references = [text.replace('_', ' ') for text in references]
#     print(f'Reference: {references[0]}')
#     base_metrics = calculate_evalute_metrics(no_penalty_predictions, references)
#     results_log.append({
#         "Base_Pen": 0.0,
#         "Freq_Pen": 0.0,
#         "Intra-Dist": base_metrics['intra_distinct'],
#         "Inter-Dist": base_metrics['inter_distinct'],
#         "BERTScore(F1)": base_metrics['bertscore']['f1'],
#         "ROUGE-L": base_metrics['rouge']['rougeL']
#     })
    
#     base_penalty = [0.2, 0.4, 0.6, 0.8, 1.0]
#     freq_penalty = [0.2, 0.4, 0.6, 0.8, 1.0] 

#     for b_penalty in base_penalty: 
#         for f_penalty in freq_penalty: 
#             penalty_tensor = generate_penalty(alpha=f_penalty)
#             penalty_predictions = model.generate_summary(batch.src, batch.src_mask, bos_idx, eos_idx, pad_idx,  max_len=360, strategy='greedy_with_penalty',penalty_tensor=penalty_tensor, base_penalty=b_penalty)
#             penalty_predictions = tokenizer.decode_batch(penalty_predictions.cpu().tolist(), skip_special_tokens=True)
#             penalty_predictions = [text.replace('_', ' ') for text in penalty_predictions]
#             metrics = calculate_evalute_metrics(penalty_predictions, references)
#             print(f'Prediction: {penalty_predictions[0]}')

#             results_log.append({
#                 "Base_Pen": b_penalty,
#                 "Freq_Pen": f_penalty,
#                 "Intra-Dist": metrics['intra_distinct'],
#                 "Inter-Dist": metrics['inter_distinct'],
#                 "BERTScore(F1)": metrics['bertscore']['f1'],
#                 "ROUGE-L": metrics['rouge']['rougeL']
#             })
            
#     df_results = pd.DataFrame(results_log)
#     df_results = df_results.sort_values(by=['Intra-Dist', 'BERTScore(F1)'], ascending=[False, False]).reset_index(drop=True)

#     return df_results

# tuning_results_df = eval_with_miniset(actual_model, batch, BOS, EOS, PAD, tokenizer)
# print("\n📊 BẢNG KẾT QUẢ GRID SEARCH:")
# display(tuning_results_df)

In [24]:
# Baseline Model In Miniset → Winner Baseline: Base=1.0, Freq=0.6 


# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.943987 	0.621122 	0.690188 	0.305074
# 1 	0.8 	1.0 	0.931421 	0.636849 	0.693590 	0.298683
# 2 	1.0 	0.8 	0.927096 	0.613263 	0.691933 	0.304440
# 3 	0.6 	1.0 	0.924895 	0.635695 	0.691410 	0.298487
# 4 	0.8 	0.8 	0.919108 	0.624915 	0.692911 	0.298059
# 5 	1.0 	0.6 	0.917390 	0.610289 	0.693766 	0.306296
# 6 	0.6 	0.8 	0.903851 	0.609536 	0.692491 	0.301862
# 7 	1.0 	0.4 	0.897232 	0.601723 	0.695358 	0.295157
# 8 	0.8 	0.6 	0.890970 	0.596078 	0.693820 	0.297927
# 9 	0.4 	1.0 	0.885245 	0.600253 	0.693475 	0.302850
# 10 	0.6 	0.6 	0.880214 	0.600770 	0.689339 	0.300759
# 11 	0.8 	0.4 	0.867442 	0.581366 	0.692033 	0.298887
# 12 	1.0 	0.2 	0.859974 	0.566645 	0.690242 	0.293723
# 13 	0.4 	0.8 	0.849245 	0.577632 	0.692294 	0.305193
# 14 	0.2 	1.0 	0.846301 	0.574270 	0.695164 	0.304163
# 15 	0.6 	0.4 	0.839201 	0.582437 	0.688064 	0.308023
# 16 	0.8 	0.2 	0.824326 	0.552831 	0.686344 	0.297474
# 17 	0.4 	0.6 	0.817345 	0.545508 	0.688793 	0.303500
# 18 	0.2 	0.8 	0.815288 	0.542668 	0.691497 	0.310027
# 19 	0.2 	0.6 	0.783578 	0.515625 	0.691227 	0.310950
# 20 	0.6 	0.2 	0.778119 	0.524030 	0.685253 	0.304452
# 21 	0.4 	0.4 	0.747817 	0.496395 	0.688354 	0.308253
# 22 	0.2 	0.4 	0.727923 	0.479280 	0.684825 	0.305066
# 23 	0.4 	0.2 	0.694555 	0.468165 	0.684532 	0.296494
# 24 	0.2 	0.2 	0.643526 	0.435718 	0.676600 	0.292410
# 25 	0.0 	0.0 	0.342083 	0.174045 	0.626877 	0.232211


# Improved Model In Miniset → Winner Improved: Base=0.8, Freq=0.8 — cân bằng tốt nhất cả 3 metric

# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.939274 	0.647887 	0.693347 	0.286116
# 1 	1.0 	0.8 	0.934163 	0.637810 	0.693827 	0.289864
# 2 	0.8 	1.0 	0.922286 	0.623769 	0.697753 	0.287233
# 3 	0.8 	0.8 	0.921282 	0.592317 	0.697531 	0.295367
# 4 	0.8 	0.6 	0.918515 	0.615385 	0.694724 	0.291566
# 5 	0.6 	1.0 	0.916468 	0.619887 	0.696884 	0.294004
# 6 	1.0 	0.6 	0.905920 	0.598406 	0.698156 	0.293007
# 7 	0.8 	0.4 	0.904266 	0.605464 	0.694333 	0.292884
# 8 	0.6 	0.8 	0.901755 	0.604545 	0.692784 	0.289050
# 9 	1.0 	0.4 	0.901273 	0.602258 	0.697079 	0.291525
# 10 	1.0 	0.2 	0.895790 	0.602378 	0.695710 	0.284385
# 11 	0.4 	1.0 	0.889800 	0.606309 	0.692071 	0.293794
# 12 	0.6 	0.6 	0.888237 	0.617647 	0.698698 	0.290871
# 13 	0.4 	0.8 	0.870922 	0.616231 	0.689821 	0.290985
# 14 	0.2 	1.0 	0.868485 	0.577273 	0.694954 	0.290945
# 15 	0.2 	0.8 	0.858090 	0.566710 	0.694390 	0.292322
# 16 	0.4 	0.6 	0.854682 	0.580397 	0.689249 	0.291973
# 17 	0.6 	0.4 	0.850639 	0.575585 	0.687068 	0.292341
# 18 	0.8 	0.2 	0.849014 	0.587097 	0.693533 	0.292755
# 19 	0.4 	0.4 	0.839190 	0.561790 	0.685476 	0.290276
# 20 	0.6 	0.2 	0.808597 	0.548616 	0.688936 	0.291172
# 21 	0.2 	0.6 	0.805314 	0.542839 	0.688385 	0.281745
# 22 	0.2 	0.4 	0.759700 	0.524335 	0.681854 	0.278744
# 23 	0.4 	0.2 	0.722275 	0.479579 	0.683564 	0.273960
# 24 	0.2 	0.2 	0.692407 	0.467895 	0.684439 	0.282220
# 25 	0.0 	0.0 	0.489910 	0.280053 	0.643636 	0.246286

# Transformer With Mamba Model In Miniset → Winner Improved: Base=1.0, Freq=0.8 

# 📊 BẢNG KẾT QUẢ GRID SEARCH:
# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.977525 	0.646851 	0.694726 	0.297507
# 1 	1.0 	0.8 	0.961424 	0.638462 	0.696308 	0.296575
# 2 	1.0 	0.6 	0.947124 	0.615220 	0.691824 	0.289222
# 3 	0.8 	1.0 	0.946854 	0.612203 	0.688966 	0.290944
# 4 	1.0 	0.4 	0.943741 	0.616005 	0.689910 	0.287144
# 5 	0.6 	1.0 	0.937250 	0.611717 	0.692151 	0.292312
# 6 	0.8 	0.6 	0.935057 	0.605007 	0.690792 	0.288516
# 7 	0.8 	0.8 	0.929264 	0.579710 	0.686932 	0.297638
# 8 	0.4 	1.0 	0.917005 	0.587690 	0.689643 	0.294436
# 9 	0.8 	0.4 	0.915731 	0.592695 	0.689119 	0.289289
# 10 	0.4 	0.8 	0.902798 	0.582080 	0.693316 	0.289192
# 11 	0.6 	0.6 	0.902698 	0.576016 	0.688671 	0.294428
# 12 	0.6 	0.8 	0.902463 	0.580128 	0.688026 	0.294928
# 13 	1.0 	0.2 	0.901849 	0.568564 	0.690879 	0.286752
# 14 	0.2 	1.0 	0.881720 	0.572003 	0.686712 	0.292176
# 15 	0.4 	0.6 	0.878596 	0.566823 	0.687670 	0.295877
# 16 	0.6 	0.4 	0.873884 	0.564721 	0.687935 	0.293123
# 17 	0.2 	0.8 	0.864156 	0.573026 	0.686132 	0.294822
# 18 	0.2 	0.6 	0.847626 	0.571708 	0.683628 	0.289174
# 19 	0.8 	0.2 	0.846419 	0.535606 	0.684588 	0.293629
# 20 	0.4 	0.4 	0.834609 	0.544567 	0.689290 	0.299222
# 21 	0.6 	0.2 	0.810290 	0.510473 	0.684643 	0.290980
# 22 	0.2 	0.4 	0.793528 	0.530888 	0.682127 	0.293942
# 23 	0.4 	0.2 	0.774203 	0.492763 	0.682379 	0.287800
# 24 	0.2 	0.2 	0.673049 	0.415250 	0.674135 	0.274830
# 25 	0.0 	0.0 	0.479597 	0.196395 	0.641298 	0.234242

# Entity Gated Mamba Model In Miniset → Winner Improved: Base=1.0, Freq=0.6
# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.916289 	0.589776 	0.696317 	0.292168
# 1 	1.0 	0.8 	0.890569 	0.555048 	0.697251 	0.304744
# 2 	0.8 	1.0 	0.879729 	0.572349 	0.693531 	0.298726
# 3 	0.6 	1.0 	0.873627 	0.557013 	0.692049 	0.301486
# 4 	1.0 	0.6 	0.871731 	0.570687 	0.694923 	0.306950
# 5 	0.8 	0.8 	0.865129 	0.568945 	0.693139 	0.304224
# 6 	0.6 	0.8 	0.861368 	0.567251 	0.688596 	0.307550
# 7 	0.4 	1.0 	0.861365 	0.557415 	0.693482 	0.308083
# 8 	0.8 	0.6 	0.853594 	0.563762 	0.689208 	0.307335
# 9 	0.6 	0.6 	0.850745 	0.571263 	0.689881 	0.296894
# 10 	0.4 	0.8 	0.850048 	0.547605 	0.692741 	0.302159
# 11 	0.2 	1.0 	0.848317 	0.547816 	0.690307 	0.302356
# 12 	1.0 	0.4 	0.847589 	0.552895 	0.690529 	0.311873
# 13 	0.2 	0.8 	0.837254 	0.545455 	0.688830 	0.298241
# 14 	0.4 	0.6 	0.835098 	0.537874 	0.692382 	0.304941
# 15 	1.0 	0.2 	0.834800 	0.561243 	0.685211 	0.298535
# 16 	0.8 	0.4 	0.834075 	0.534368 	0.690271 	0.303036
# 17 	0.8 	0.2 	0.827480 	0.546045 	0.686149 	0.291831
# 18 	0.4 	0.4 	0.811815 	0.524501 	0.687850 	0.300106
# 19 	0.6 	0.4 	0.805442 	0.526490 	0.686393 	0.296362
# 20 	0.6 	0.2 	0.786904 	0.526578 	0.691733 	0.300501
# 21 	0.2 	0.6 	0.784871 	0.494743 	0.688892 	0.303565
# 22 	0.2 	0.4 	0.740223 	0.479885 	0.684615 	0.294363
# 23 	0.4 	0.2 	0.739748 	0.496648 	0.680052 	0.302548
# 24 	0.2 	0.2 	0.684422 	0.461209 	0.678902 	0.288243
# 25 	0.0 	0.0 	0.510402 	0.290045 	0.652972 	0.259374


# Transformer With HyperSphereNorm, Weight Tying, And Label Smoothing Model In Miniset → Winner Improved: Base=0.2, Freq=0.4
# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.855036 	0.716958 	0.750367 	0.385104
# 1 	0.4 	0.6 	0.852151 	0.713303 	0.752096 	0.389567
# 2 	1.0 	0.8 	0.848605 	0.713510 	0.749495 	0.387311
# 3 	0.6 	0.8 	0.846386 	0.711406 	0.752031 	0.393857
# 4 	0.2 	0.8 	0.844159 	0.706183 	0.751445 	0.393044
# 5 	0.2 	0.4 	0.841577 	0.697294 	0.758399 	0.406917
# 6 	0.2 	0.2 	0.839874 	0.698157 	0.756041 	0.403145
# 7 	0.6 	1.0 	0.839686 	0.693958 	0.754843 	0.392392
# 8 	0.6 	0.4 	0.838081 	0.702811 	0.751163 	0.388193
# 9 	0.8 	0.2 	0.837437 	0.709677 	0.749850 	0.388319
# 10 	0.6 	0.2 	0.837363 	0.702532 	0.752209 	0.389319
# 11 	0.8 	1.0 	0.837001 	0.694381 	0.747201 	0.387154
# 12 	0.4 	1.0 	0.836820 	0.702874 	0.752725 	0.392627
# 13 	0.2 	1.0 	0.834047 	0.694568 	0.747983 	0.389821
# 14 	0.6 	0.6 	0.834046 	0.702609 	0.751686 	0.390021
# 15 	0.4 	0.8 	0.833140 	0.692948 	0.750536 	0.390428
# 16 	0.4 	0.4 	0.832260 	0.699885 	0.751422 	0.388475
# 17 	0.8 	0.6 	0.831157 	0.687816 	0.751109 	0.391906
# 18 	0.2 	0.6 	0.828647 	0.691584 	0.750433 	0.399489
# 19 	1.0 	0.6 	0.827323 	0.689675 	0.749491 	0.385867
# 20 	0.8 	0.4 	0.823408 	0.686508 	0.749001 	0.391505
# 21 	1.0 	0.4 	0.822763 	0.681343 	0.750748 	0.392194
# 22 	0.4 	0.2 	0.822037 	0.688534 	0.751740 	0.396221
# 23 	0.8 	0.8 	0.819313 	0.683240 	0.750236 	0.395658
# 24 	1.0 	0.2 	0.816683 	0.678492 	0.747604 	0.391084
# 25 	0.0 	0.0 	0.813345 	0.666846 	0.752727 	0.406869

# Entity Gated Mamba With Lable Smoothing In Miniset → Winner Improved: Base = 0.8, Freq = 0.8

# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.873183 	0.592937 	0.690263 	0.311041
# 1 	0.8 	1.0 	0.850493 	0.562237 	0.689964 	0.312485
# 2 	1.0 	0.8 	0.843498 	0.560263 	0.688873 	0.304414
# 3 	0.6 	1.0 	0.840344 	0.554980 	0.690832 	0.313502
# 4 	0.4 	1.0 	0.824639 	0.539988 	0.688733 	0.307902
# 5 	0.8 	0.8 	0.822392 	0.557882 	0.692128 	0.316074
# 6 	1.0 	0.4 	0.821039 	0.548349 	0.691403 	0.310291
# 7 	0.4 	0.8 	0.818957 	0.546341 	0.690431 	0.307350
# 8 	1.0 	0.6 	0.814142 	0.550548 	0.692987 	0.310543
# 9 	0.6 	0.8 	0.811263 	0.540701 	0.686771 	0.301840
# 10 	0.6 	0.6 	0.804433 	0.549912 	0.685804 	0.306219
# 11 	0.2 	1.0 	0.803414 	0.533531 	0.688297 	0.306189
# 12 	0.8 	0.4 	0.803133 	0.532802 	0.690609 	0.306073
# 13 	0.8 	0.6 	0.798379 	0.523451 	0.691035 	0.310985
# 14 	0.4 	0.6 	0.793124 	0.526531 	0.690138 	0.308420
# 15 	0.6 	0.4 	0.792980 	0.538122 	0.690767 	0.305957
# 16 	0.2 	0.8 	0.790179 	0.527372 	0.689775 	0.303120
# 17 	0.8 	0.2 	0.786534 	0.526220 	0.691203 	0.306315
# 18 	0.4 	0.4 	0.776555 	0.520973 	0.688394 	0.299237
# 19 	1.0 	0.2 	0.768954 	0.526731 	0.689605 	0.308568
# 20 	0.2 	0.6 	0.761866 	0.525785 	0.689425 	0.309207
# 21 	0.6 	0.2 	0.751857 	0.503708 	0.691456 	0.301396
# 22 	0.4 	0.2 	0.710451 	0.478944 	0.682262 	0.295667
# 23 	0.2 	0.4 	0.707694 	0.470356 	0.688021 	0.305332
# 24 	0.2 	0.2 	0.701374 	0.465828 	0.687295 	0.303234
# 25 	0.0 	0.0 	0.638173 	0.452199 	0.680302 	0.301923

# Entity Guided Hybrid Mamba In Miniset → Winner Improved: Base = 1.0, Freq = 1.0
# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	1.0 	1.0 	0.881038 	0.727382 	0.749481 	0.377860
# 1 	1.0 	0.8 	0.875726 	0.711009 	0.745008 	0.375932
# 2 	0.8 	0.8 	0.874028 	0.721564 	0.740464 	0.374295
# 3 	0.6 	0.8 	0.872424 	0.714373 	0.742244 	0.368915
# 4 	1.0 	0.6 	0.872264 	0.722322 	0.743960 	0.373402
# 5 	0.8 	0.6 	0.866885 	0.715406 	0.745334 	0.380121
# 6 	0.6 	1.0 	0.865710 	0.707975 	0.740972 	0.368454
# 7 	0.8 	1.0 	0.860546 	0.711584 	0.743579 	0.378845
# 8 	0.4 	1.0 	0.858958 	0.692890 	0.738351 	0.368670
# 9 	0.4 	0.8 	0.857899 	0.695824 	0.741681 	0.378613
# 10 	1.0 	0.4 	0.857002 	0.696919 	0.740372 	0.367365
# 11 	0.8 	0.2 	0.847998 	0.690133 	0.740760 	0.377765
# 12 	0.8 	0.4 	0.847749 	0.687717 	0.741763 	0.375477
# 13 	0.6 	0.6 	0.845619 	0.686628 	0.743714 	0.370345
# 14 	0.4 	0.6 	0.840995 	0.698009 	0.743701 	0.377519
# 15 	0.2 	1.0 	0.835418 	0.688696 	0.742399 	0.372589
# 16 	1.0 	0.2 	0.832208 	0.680418 	0.740153 	0.374392
# 17 	0.2 	0.8 	0.828817 	0.682339 	0.741932 	0.380141
# 18 	0.2 	0.6 	0.821464 	0.670504 	0.740026 	0.376326
# 19 	0.6 	0.4 	0.819219 	0.668208 	0.739620 	0.378786
# 20 	0.4 	0.4 	0.816542 	0.668464 	0.738646 	0.372890
# 21 	0.2 	0.4 	0.816000 	0.672113 	0.740100 	0.376311
# 22 	0.6 	0.2 	0.813545 	0.674930 	0.739338 	0.372806
# 23 	0.2 	0.2 	0.805468 	0.653304 	0.738322 	0.372229
# 24 	0.4 	0.2 	0.793194 	0.643269 	0.740265 	0.365391
# 25 	0.0 	0.0 	0.725654 	0.626752 	0.725006 	0.343826

# Entity Guided Pure Transformer In Miniset → Winner Improved: Base = 0.6, Freq = 1.0
# 📊 BẢNG KẾT QUẢ GRID SEARCH:

# 	Base_Pen 	Freq_Pen 	Intra-Dist 	Inter-Dist 	BERTScore(F1) 	ROUGE-L
# 0 	0.6 	1.0 	0.879573 	0.740576 	0.761776 	0.399735
# 1 	0.6 	0.8 	0.872822 	0.726087 	0.756547 	0.395547
# 2 	0.8 	1.0 	0.869647 	0.738109 	0.760277 	0.400276
# 3 	1.0 	1.0 	0.868988 	0.714747 	0.758285 	0.390125
# 4 	0.8 	0.6 	0.865771 	0.727978 	0.758637 	0.399484
# 5 	0.8 	0.8 	0.863927 	0.722132 	0.760212 	0.395857
# 6 	1.0 	0.8 	0.851501 	0.712480 	0.758199 	0.391937
# 7 	0.8 	0.4 	0.850491 	0.717226 	0.757264 	0.395434
# 8 	1.0 	0.2 	0.848680 	0.719540 	0.759738 	0.395011
# 9 	0.4 	1.0 	0.848348 	0.714819 	0.756844 	0.387396
# 10 	1.0 	0.4 	0.844670 	0.711421 	0.760306 	0.395106
# 11 	1.0 	0.6 	0.843704 	0.706787 	0.760138 	0.398871
# 12 	0.6 	0.4 	0.839435 	0.710784 	0.753431 	0.389793
# 13 	0.4 	0.4 	0.830376 	0.702791 	0.750586 	0.385410
# 14 	0.6 	0.6 	0.828781 	0.690476 	0.746491 	0.377927
# 15 	0.8 	0.2 	0.828485 	0.699297 	0.749888 	0.383184
# 16 	0.4 	0.6 	0.826805 	0.696477 	0.746828 	0.375620
# 17 	0.2 	1.0 	0.825315 	0.696300 	0.752847 	0.385831
# 18 	0.2 	0.6 	0.819798 	0.694906 	0.748669 	0.390127
# 19 	0.4 	0.8 	0.812955 	0.694640 	0.749682 	0.384057
# 20 	0.4 	0.2 	0.811306 	0.682480 	0.749595 	0.385101
# 21 	0.6 	0.2 	0.805939 	0.681818 	0.747250 	0.382287
# 22 	0.2 	0.8 	0.804440 	0.681583 	0.751199 	0.382860
# 23 	0.2 	0.4 	0.798716 	0.677473 	0.749548 	0.390892
# 24 	0.2 	0.2 	0.778411 	0.657594 	0.746079 	0.387113
# 25 	0.0 	0.0 	0.732181 	0.600565 	0.747824 	0.392989

In [25]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [26]:
seed(42)
import os
import gc
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu' 
D_MODEL = 512
D_FF = 2048
H = 8
N = 6

BOS = tokenizer.token_to_id('<BOS>')
EOS = tokenizer.token_to_id('<EOS>')
PAD = tokenizer.token_to_id('<PAD>')

test_dataloader = DataLoader(test_dataset, 
                                 batch_size=32, 
                                 shuffle=False, 
                                 collate_fn=collate_fn, 
                                 pin_memory=False, 
                                 persistent_workers=False,
                                 num_workers=0)

# -----------------------------------------------------------------
# KHỞI TẠO VÀ NẠP TRỌNG SỐ CHO 8 MÔ HÌNH (ĐÃ SỬA TÊN BIẾN PENALTY)
# -----------------------------------------------------------------

# Model 1
model_1 = BaselineTransformer(D_MODEL, D_FF, H, N)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/improve-baseline-tranformers-results/baseline_transformer/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/improve-baseline-tranformers-results/baseline_transformer/best_checkpoint.pt', map_location=device)
    model_1.load_state_dict(checkpoint['model_state_dict'])
model_1 = model_1.to(device)
base_penalty_1 = 1.0 
freq_penalty_1 = 0.6
penalty_tensor_1 = generate_penalty(freq_penalty_1) # 🔥 Fix: Đổi tên thành penalty_tensor_1

# Model 2
D_FF_MODEL_2 = 1365 
model_2 = ImprovedBaselineTransformer(D_MODEL, D_FF_MODEL_2, H, N)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/improve-baseline-tranformers-results/improved_baseline_transformer/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/improve-baseline-tranformers-results/improved_baseline_transformer/best_checkpoint.pt', map_location=device)
    model_2.load_state_dict(checkpoint['model_state_dict'])    
model_2 = model_2.to(device)
base_penalty_2 = 0.8
freq_penalty_2 = 0.8
penalty_tensor_2 = generate_penalty(freq_penalty_2)

# Model 3
base_penalty_3 = 1.0
freq_penalty_3 = 0.8
D_FF_NEW_MODEL_3 = 798
model_3 = TransformerWithSoftPromptMamba(D_MODEL, D_FF_MODEL_2, D_FF_NEW_MODEL_3, H, N)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/improve-with-mamba-and-hyperspherenorm-results/transformer_with_soft_prompt_mamba/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/improve-with-mamba-and-hyperspherenorm-results/transformer_with_soft_prompt_mamba/best_checkpoint.pt', map_location=device)
    model_3.load_state_dict(checkpoint['model_state_dict'])
model_3 = model_3.to(device)
penalty_tensor_3 = generate_penalty(freq_penalty_3)

# Model 4
base_penalty_4 = 1.0
freq_penalty_4 = 0.8
model_4 = EntityGatedMambaSeq2Seq(D_MODEL, D_FF_MODEL_2, D_FF_MODEL_2, H, N + 4)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/entity-gated-mamba-seq2seq-results/entity_gated_mamba_seq2Seq/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/entity-gated-mamba-seq2seq-results/entity_gated_mamba_seq2Seq/best_checkpoint.pt', map_location=device)
    model_4.load_state_dict(checkpoint['model_state_dict'])
model_4 = model_4.to(device)
penalty_tensor_4 = generate_penalty(freq_penalty_4)

# Model 5
base_penalty_5 = 0.2
freq_penalty_5 = 0.4
model_5 = TransformerWithWeightTyingAndHyperSphereNorm(D_MODEL, D_FF_MODEL_2, H, N + 5)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/transformer-with-weight-tying-and-hyperspherenorm/transformer_with_hyperspherenorm_and_weight_tying/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/transformer-with-weight-tying-and-hyperspherenorm/transformer_with_hyperspherenorm_and_weight_tying/best_checkpoint.pt', map_location=device)
    model_5.load_state_dict(checkpoint['model_state_dict'])
model_5 = model_5.to(device)
penalty_tensor_5 = generate_penalty(freq_penalty_5) # 🔥 Fix: Tránh trùng lặp biến toàn cục

# Model 6
base_penalty_6 = 0.8
freq_penalty_6 = 0.8
model_6 = EntityGatedMambaSeq2Seq(D_MODEL, D_FF_MODEL_2, D_FF_MODEL_2, H, N + 4)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/entity-gated-mamba-label-smoothing-results/entity_gated_mamba_label_smoothing/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/entity-gated-mamba-label-smoothing-results/entity_gated_mamba_label_smoothing/best_checkpoint.pt', map_location=device)
    model_6.load_state_dict(checkpoint['model_state_dict'])
model_6 = model_6.to(device)
penalty_tensor_6 = generate_penalty(freq_penalty_6)

# Model 7
base_penalty_7 = 1.0
freq_penalty_7 = 1.0
model_7 = EntityGuidedHybridMamba(D_MODEL, D_FF_MODEL_2, D_FF_MODEL_2, H, N+4)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/entity-guided-hybrid-mamba-results/entity_guided_hybrid_mamba/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/entity-guided-hybrid-mamba-results/entity_guided_hybrid_mamba/best_checkpoint.pt', map_location=device)
    model_7.load_state_dict(checkpoint['model_state_dict'])
model_7 = model_7.to(device)
penalty_tensor_7 = generate_penalty(freq_penalty_7)

# Model 8
base_penalty_8 = 0.6
freq_penalty_8 = 1.0
model_8 = EntityGuidedPureTransformer(D_MODEL, D_FF_MODEL_2, D_FF_MODEL_2, H, N+5)
if os.path.exists('/kaggle/input/datasets/longnguyen2k5/entity-guided-pure-transformer-results/entity_guided_pure_transformer/best_checkpoint.pt'): 
    checkpoint = torch.load('/kaggle/input/datasets/longnguyen2k5/entity-guided-pure-transformer-results/entity_guided_pure_transformer/best_checkpoint.pt', map_location=device)
    model_8.load_state_dict(checkpoint['model_state_dict'])
model_8 = model_8.to(device)
penalty_tensor_8 = generate_penalty(freq_penalty_8)

# -----------------------------------------------------------------
# CẤU HÌNH PIPELINE VÒNG LẶP GOM KẾT QUẢ (ĐÃ FIX CHÍNH TẢ LABLE)
# -----------------------------------------------------------------
models_pipeline = [
    ("Baseline Transformer", model_1, penalty_tensor_1, base_penalty_1),
    ("Improved Baseline", model_2, penalty_tensor_2, base_penalty_2),
    ("Soft Prompt Mamba", model_3, penalty_tensor_3, base_penalty_3),
    ("Entity Gated Mamba", model_4, penalty_tensor_4, base_penalty_4),
    ("Transformer Hypersphere", model_5, penalty_tensor_5, base_penalty_5),
    ("Entity Gated Mamba With Label Smoothing", model_6, penalty_tensor_6, base_penalty_6), # 🔥 Fix chính tả thành 'Label'
    ("Entity Guided Hybrid Mamba", model_7, penalty_tensor_7, base_penalty_7),
    ("Entity Guided Pure Transformer", model_8, penalty_tensor_8, base_penalty_8)
]

master_dictionary = {}
is_metadata_saved = False

for model_name, model_obj, p_tensor, b_pen in models_pipeline:
    print(f"\n" + "="*60)
    print(f"🔮 ĐANG KHỞI CHẠY HỆ THỐNG ĐÁNH GIÁ CHO: {model_name.upper()}")
    print("="*60)
    
    # LẦN CHẠY 1: GREEDY GỐC (VANILLA)
    print(f"🔹 Lượt 1: Chạy chiến lược Greedy Gốc...")
    sources_vanilla, references_vanilla, predictions_vanilla = evaluate_model(
        model=model_obj,
        test_dataloader=test_dataloader,
        bos_idx=BOS, eos_idx=EOS, pad_idx=PAD,
        tokenizer=tokenizer,
        strategy='greedy',
        penalty_tensor=None,
        base_penalty=0.0
    )
    
    if not is_metadata_saved:
        master_dictionary['source'] = sources_vanilla
        master_dictionary['reference'] = references_vanilla
        is_metadata_saved = True
        
    master_dictionary[f"{model_name} (Vanilla)"] = predictions_vanilla
    
    # LẦN CHẠY 2: GREEDY + GTF-PENALTY
    print(f"🔸 Lượt 2: Chạy chiến lược Greedy + GTF-Penalty...")
    _, _, predictions_penalty = evaluate_model(
        model=model_obj,
        test_dataloader=test_dataloader,
        bos_idx=BOS, eos_idx=EOS, pad_idx=PAD,
        tokenizer=tokenizer,
        strategy='greedy_with_penalty',
        penalty_tensor=p_tensor,
        base_penalty=b_pen
    )
    
    master_dictionary[f"{model_name} (Penalty)"] = predictions_penalty
    
    # GIẢI PHÓNG VRAM
    del model_obj
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ Đã giải phóng hoàn toàn bộ nhớ và VRAM GPU của {model_name}.")

# Xuất file CSV Master duy nhất
df_master_final = pd.DataFrame(master_dictionary)
df_master_final.to_csv('/kaggle/working/master_evaluation_dataset.csv', index=False)
print("\n" + "=*="*20)
print("🎉 HOÀN THÀNH TOÀN BỘ PIPELINE! File master_evaluation_dataset.csv đã sẵn sàng!")
print("=*="*20)

Done


100%|██████████| 674/674 [00:02<00:00, 295.23it/s]



🔮 ĐANG KHỞI CHẠY HỆ THỐNG ĐÁNH GIÁ CHO: BASELINE TRANSFORMER
🔹 Lượt 1: Chạy chiến lược Greedy Gốc...


[GENERATING PREDICTIONS]: 100%|██████████| 42/42 [03:13<00:00,  4.61s/it]


🔸 Lượt 2: Chạy chiến lược Greedy + GTF-Penalty...


...

✅ Đã giải phóng hoàn toàn bộ nhớ và VRAM GPU của Entity Guided Hybrid Mamba.

🔮 ĐANG KHỞI CHẠY HỆ THỐNG ĐÁNH GIÁ CHO: ENTITY GUIDED PURE TRANSFORMER
🔹 Lượt 1: Chạy chiến lược Greedy Gốc...


[GENERATING PREDICTIONS]: 100%|██████████| 42/42 [03:28<00:00,  4.97s/it]


🔸 Lượt 2: Chạy chiến lược Greedy + GTF-Penalty...


[GENERATING PREDICTIONS]: 100%|██████████| 42/42 [02:24<00:00,  3.45s/it]


✅ Đã giải phóng hoàn toàn bộ nhớ và VRAM GPU của Entity Guided Pure Transformer.

=*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*=
🎉 HOÀN THÀNH TOÀN BỘ PIPELINE! File master_evaluation_dataset.csv đã sẵn sàng!
=*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*==*=


In [27]:
from torchinfo import summary
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu' 
D_MODEL = 512
H = 8
N = 6
D_FF = 1365 
D_FF_NEW = 798

# 2. Wrapper thông minh của bạn (Giữ nguyên)
class ModelWrapper(nn.Module):
    def __init__(self, m): 
        super().__init__()
        self.m = m
    def forward(self, *args, **kwargs): 
        return self.m(*args, **kwargs)[0]
def count_params(model):
    # Tổng (cách torchinfo đếm - dùng để debug)
    total = sum(p.numel() for p in model.parameters())
    
    # Unique (dùng để so sánh công bằng)
    unique = sum(p.numel() for p in {p.data_ptr(): p 
                                      for p in model.parameters()}.values())
    
    print(f"Total (torchinfo style): {total:,}")
    print(f"Unique (fair comparison): {unique:,}")
    print(f"Saved by weight tying:   {total - unique:,}")

count_params(model_1)
count_params(model_2)
count_params(model_3)
count_params(model_4)
count_params(model_5)
count_params(model_6)
count_params(model_7)
count_params(model_8)

Total (torchinfo style): 99,470,496
Unique (fair comparison): 99,470,496
Saved by weight tying:   0
Total (torchinfo style): 99,418,272
Unique (fair comparison): 99,418,272
Saved by weight tying:   0
Total (torchinfo style): 99,418,117
Unique (fair comparison): 99,418,117
Saved by weight tying:   0
Total (torchinfo style): 92,815,877
Unique (fair comparison): 92,815,877
Saved by weight tying:   0
Total (torchinfo style): 99,256,833
Unique (fair comparison): 99,256,833
Saved by weight tying:   0
Total (torchinfo style): 92,815,877
Unique (fair comparison): 92,815,877
Saved by weight tying:   0
Total (torchinfo style): 92,816,386
Unique (fair comparison): 92,816,386
Saved by weight tying:   0
Total (torchinfo style): 99,257,858
Unique (fair comparison): 99,257,858
Saved by weight tying:   0
